In [ ]:


export=False


## Packages ---

# General
import numpy as np
import pandas as pd
import os
from pathlib import Path
import getpass
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format



## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'

path_plots  = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
path_access = Path(r'I:\Projects\Josh\Regional Monitoring\Accessibility')



## User defined functions ---

path_func = path_config0 / 'Functions.py'

with path_func.open("r") as f:
    exec(f.read())


def proc_access_1(access, metric, geography, col_name):

    if geography == 'Community Type':
        file_name = f"Community_Type_2024_dissolve__access_pop_{access}.csv"
    if geography == 'Jurisdiction':
        file_name = f"City_County__access_pop_{access}.csv"
    if geography == 'County':
        file_name = f"tl_2020_sacog_county__access_pop_{access}.csv"
    if geography == 'MPO':
        file_name = f"SACOG_MPO__access_pop_{access}.csv"

    file_in = path_access / file_name
    df = pd.read_csv(file_in)

    if access=='emp':
        cols_access = [f'drive_{access}', f'transit_{access}', f'bike_{access}', f'walk_{access}']
    else:
        cols_access = [f'transit_{access}', f'bike_{access}', f'walk_{access}']

    df = df.rename(columns={col_name:geography})
    df = df[[geography] + cols_access]
    df = df.groupby(geography, as_index=False).mean() # soon to be weighted average by pop in comm type
    df = df.melt(id_vars=geography, var_name='mode', value_name=metric)

    df['Sort_mode'] = pd.Categorical(df['mode'], cols_access)
    df['Sort'] = df.groupby(geography, as_index=False)[metric].transform('sum')
    df = df.sort_values(['Sort', 'Sort_mode'], ascending=[True,True])
    df = df.drop(['Sort_mode', 'Sort'], axis=1)

    if geography == 'Jurisdiction':
        df = df[df[geography] != 'South Lake Tahoe']
        df.loc[df[geography].str.contains('County'), geography] = df[df[geography].str.contains('County')][geography] + ' Unincorporated'

    df = df.reset_index(drop=True)

    if access=='emp':
        conditions = [
            df['mode'] == f'transit_{access}'
            , df['mode'] == f'bike_{access}'
            , df['mode'] == f'walk_{access}'
            , df['mode'] == f'drive_{access}'
        ]
        choices = ['Public Transit', 'Bike', 'Walk', 'Drive']
    else:
        conditions = [
            df['mode'] == f'transit_{access}'
            , df['mode'] == f'bike_{access}'
            , df['mode'] == f'walk_{access}'
        ]
        choices = ['Public Transit', 'Bike', 'Walk']

    df['mode'] = np.select(conditions, choices, default='no')

    return df


def proc_access_2(access, metric, geography, col_name):

    dict_race_eth = {'asian': 'Asian (NH)', 'black':'Black or African American (NH)', 'white':'White (NH)', 'hispanic':'Hispanic or Latino'}

    if geography == 'MPO':
        files = [f"SACOG_MPO__access_asian_{access}.csv", f"SACOG_MPO__access_black_{access}.csv", f"SACOG_MPO__access_white_{access}.csv", f"SACOG_MPO__access_hispanic_{access}.csv"]
    if geography == 'County':
        files = [f"tl_2020_sacog_county__access_asian_{access}.csv", f"tl_2020_sacog_county__access_black_{access}.csv", f"tl_2020_sacog_county__access_white_{access}.csv", f"tl_2020_sacog_county__access_hispanic_{access}.csv"]
    if geography == 'Jurisdiction':
        files = [f"City_County__access_asian_{access}.csv", f"City_County__access_black_{access}.csv", f"City_County__access_white_{access}.csv", f"City_County__access_hispanic_{access}.csv"]
    if geography == 'Community Type':
        files = [f"Community_Type_2024_dissolve__access_asian_{access}.csv", f"Community_Type_2024_dissolve__access_black_{access}.csv", f"Community_Type_2024_dissolve__access_white_{access}.csv", f"Community_Type_2024_dissolve__access_hispanic_{access}.csv"]

    list_df = []
    for file in files:
        
        file_in = path_access / file
        df = pd.read_csv(file_in)

        for race_eth_code, race_eth_full in dict_race_eth.items():
            if race_eth_code in file:
                df['Race_Ethnicity'] = race_eth_full

        list_df.append(df)

    df = pd.concat(list_df)

    if access=='emp':
        cols_access = [f'drive_{access}', f'transit_{access}', f'bike_{access}', f'walk_{access}']
    else:
        cols_access = [f'transit_{access}', f'bike_{access}', f'walk_{access}']

    df = df.rename(columns={col_name:geography})
    df = df[[geography, 'Race_Ethnicity'] + cols_access]
    df = df.melt(id_vars=[geography, 'Race_Ethnicity'], var_name='mode', value_name=metric)

    df['Sort_mode'] = pd.Categorical(df['mode'], cols_access)
    df['Sort_eth' ] = pd.Categorical(df['Race_Ethnicity'], ['Asian (NH)', 'Black or African American (NH)', 'Hispanic or Latino', 'White (NH)'])
    df['Sort'] = df.groupby(geography, as_index=False)[metric].transform('sum')
    df = df.sort_values(['Sort', 'Sort_mode', 'Sort_eth'], ascending=[False,True, False])
    df = df.drop(['Sort_eth', 'Sort_mode', 'Sort'], axis=1)
    df = df.reset_index(drop=True)

    if access=='emp':
        conditions = [
            df['mode'] == f'transit_{access}'
            , df['mode'] == f'bike_{access}'
            , df['mode'] == f'walk_{access}'
            , df['mode'] == f'drive_{access}'
        ]
        choices = ['Public Transit', 'Bike', 'Walk', 'Drive']
    else:
        conditions = [
            df['mode'] == f'transit_{access}'
            , df['mode'] == f'bike_{access}'
            , df['mode'] == f'walk_{access}'
        ]
        choices = ['Public Transit', 'Bike', 'Walk']

    df['mode'] = np.select(conditions, choices, default='no')

    return df



def plot_agol(fig, indicator, title, template, font_family, config, plot_name, export, font_size):
    fig.update_layout(
        legend_title=None
        , title=title
        , template=template
        , font_family=font_family
        , xaxis_title=None
        , yaxis_title=None
        , yaxis=dict(tickfont=dict(size=font_size))
        , xaxis=dict(tickfont=dict(size=font_size))
        )
    
    fig.show(config=config)
    
    if export:
        fig.write_html( file=os.path.join(path_plots, f'{indicator}_{plot_name}.html'), config=config)
        


def plot_access_1(df_plot, indicator, access, metric, geography, export, font_size):

    plot_name = f'{access}_{re.sub(' ', '_', geography.lower())}'

    color_map = {
       'Walk':'#FBB117'
       , 'Bike': '#9DC209'
       , 'Public Transit':"#1E90FF"
       , 'Drive':"#1F45FC"
    }

    if geography == 'MPO':
        df_plot.loc[df_plot['MPO'].str.contains('Sacramento'), 'MPO'] = 'SACOG'

    fig = px.bar(df_plot, x=metric, y=geography, orientation='h'
                , color='mode'
                , color_discrete_map=color_map)

    title = f'<b>{metric} Accessible by Modes of Transportation by {geography}</b>'
    fig.update_traces(hovertemplate='%{x}')
    fig.update_layout(legend={'traceorder': 'reversed'})
    fig.update_xaxes(tick0=0, tickformat=',.0f')

    fig.add_annotation(
        x=0.9, y=-0.1,
        text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
        showarrow=False,
        xanchor="left", 
        yanchor="top",
        font=dict(size=10),
        xref="paper", yref="paper"
    )

    font_family = 'Microsoft YaHei'
    template = 'plotly_white'
    config={'modeBarButtonsToRemove': ['select', 'lasso', 'toImage'], 'displaylogo': False}

    plot_agol(fig, indicator, title, template, font_family, config, plot_name, export, font_size)



def plot_access_2(df_plot, indicator, access, metric, geography, export, font_size):

    plot_name = f'{access}_{re.sub(' ', '_', geography.lower())}'

    color_map = {
       'Walk':'#FBB117'
       , 'Bike': '#9DC209'
       , 'Public Transit':"#1E90FF"
       , 'Drive':"#1F45FC"
    }

    if geography == 'MPO':
        df_plot.loc[df_plot['MPO'].str.contains('Sacramento'), 'MPO'] = 'SACOG'
    if geography == 'Community Type':
        df_plot.loc[df_plot['Community Type'] == 'Centers and Corridors'         , 'Community Type'] = 'Centers and<br>Corridors'
        df_plot.loc[df_plot['Community Type'] == 'Established Communities'       , 'Community Type'] = 'Established<br>Communities'
        df_plot.loc[df_plot['Community Type'] == 'Not Identified for Growth'     , 'Community Type'] = 'Not Identified<br>for Growth'
        df_plot.loc[df_plot['Community Type'] == 'Agricultural and Natural Lands', 'Community Type'] = 'Agricultural and<br>Natural Lands'
        df_plot.loc[df_plot['Community Type'] == 'Developing Communities'        , 'Community Type'] = 'Developing<br>Communities'

    fig = px.bar(df_plot, x=metric, y='Race_Ethnicity', orientation='h'
                    , color='mode'
                    , color_discrete_map=color_map
                    , facet_row = geography)

    if geography != 'MPO':
        title = f'<b>{metric} Accessible by Modes of Transportation by Race/Ethnicity by {geography}</b>'
    if geography == 'MPO':
        title = f'<b>{metric} Accessible by Modes of Transportation by Race/Ethnicity</b>'
    fig.update_traces(hovertemplate='%{x:,.0f}')
    fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.1,xanchor="right", x=0.225))
    fig.for_each_yaxis(lambda y: y.update(title=''))
    fig.for_each_xaxis(lambda x: x.update(tick0=0, tickformat=',.0f'))
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[1]))
    for annotation in fig['layout']['annotations']:
        annotation['textangle']=0

    fig.add_annotation(
        x=0.85, y=-0.05,
        text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
        showarrow=False,
        xanchor="left", 
        yanchor="top",
        font=dict(size=10),
        xref="paper", yref="paper"
    )

    font_family = 'Microsoft YaHei'
    template = 'plotly_white'
    config={'modeBarButtonsToRemove': ['select', 'lasso', 'toImage'], 'displaylogo': False}

    plot_agol(fig, indicator, title, template, font_family, config, plot_name, export, font_size)




In [ ]:

dict_access = {'emp':'Number of Jobs', 'edu':'Number of Schools', 'nonwork':'Number of Neighborhood Services'}


Travel_Access_1

In [ ]:


indicator = 'TravelAccess_1'
geography = 'Community Type'
col_name = 'Webmap_ComType'
font_size = 12

for access, metric in dict_access.items():

    df_plot = proc_access_1(access, metric, geography, col_name)
    display(df_plot.head())
    plot_access_1(df_plot, indicator, access, metric, geography, export, font_size)



In [ ]:


indicator = 'TravelAccess_1'
geography = 'Jurisdiction'
col_name = 'JURIS'
font_size = 9

for access, metric in dict_access.items():

    df_plot = proc_access_1(access, metric, geography, col_name)
    display(df_plot.head())
    plot_access_1(df_plot, indicator, access, metric, geography, export, font_size)



In [ ]:


indicator = 'TravelAccess_1'
geography = 'County'
col_name = 'name'
font_size = 12

for access, metric in dict_access.items():

    df_plot = proc_access_1(access, metric, geography, col_name)
    display(df_plot.head())
    plot_access_1(df_plot, indicator, access, metric, geography, export, font_size)



In [ ]:


indicator = 'TravelAccess_1'
geography = 'MPO'
col_name = 'MPO_NAME'
font_size = 12

for access, metric in dict_access.items():

    df_plot = proc_access_1(access, metric, geography, col_name)
    display(df_plot.head())
    plot_access_1(df_plot, indicator, access, metric, geography, export, font_size)



Travel_Access_2

In [ ]:


indicator = 'TravelAccess_2'
geography = 'Community Type'
col_name = 'Webmap_ComType'
font_size = 12

for access, metric in dict_access.items():

    df_plot = proc_access_2(access, metric, geography, col_name)
    display(df_plot.head())
    plot_access_2(df_plot, indicator, access, metric, geography, export, font_size)



In [ ]:

## Jurisdiction a little wonky, too many geographies
# indicator = 'TravelAccess_2'
# geography = 'Jurisdiction'
# col_name = 'JURIS'
# font_size = 12

# for access, metric in dict_access.items():

#     df_plot = proc_access_2(access, metric, geography, col_name)
#     display(df_plot.head())
#     plot_access_2(df_plot, indicator, access, metric, geography, export, font_size)
   


In [ ]:


indicator = 'TravelAccess_2'
geography = 'County'
col_name = 'name'
font_size = 12

for access, metric in dict_access.items():

    df_plot = proc_access_2(access, metric, geography, col_name)
    display(df_plot.head())
    plot_access_2(df_plot, indicator, access, metric, geography, export, font_size)



In [ ]:


indicator = 'TravelAccess_2'
geography = 'MPO'
col_name = 'MPO_NAME'
font_size = 12

for access, metric in dict_access.items():

    df_plot = proc_access_2(access, metric, geography, col_name)
    display(df_plot.head())
    plot_access_2(df_plot, indicator, access, metric, geography, export, font_size)



In [ ]:


# # Set Indicator
# indicator_name = 'TravelAccess_2'
# geography = 'MPO_NAME'


# color_map = {
#        'Walk':'#FBB117'
#        , 'Bike': '#9DC209'
#        , 'Public Transit':"#1E90FF"
#        , 'Drive':"#1F45FC"
# }


# ## Number of Jobs ---

# access = 'emp'
# metric = 'number_of_jobs'
# plot_title = 'Number of Jobs'
# plot_name = f'{access}_race_eth'

# # Organizing
# df_plot = access_2_mpo(access, metric, geography)
# display(df_plot.head())


# # Plotting
# fig = px.bar(df_plot, y='Race_Ethnicity', x=metric
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Race/Ethnicity</b>'
# fig.update_traces(hovertemplate='%{x}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=25000, range=[0, 255000], tickformat=',.0f')


# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left", 
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



# ## Number of Schools ---

# access = 'edu'
# metric = 'number_of_schools'
# plot_title = 'Number of Schools'
# plot_name = f'{access}_race_eth'

# # Organizing
# df_plot = access_2_mpo(access, metric, geography)
# display(df_plot.head())


# # Plotting
# fig = px.bar(df_plot, y='Race_Ethnicity', x=metric
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Race/Ethnicity</b>'
# fig.update_traces(hovertemplate='%{x}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=5, range=[0, 46], tickformat=',.0f')


# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left", 
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)




# ## Number of Neighborhood Services ---

# access = 'nonwork'
# metric = 'number_of_nonwork'
# plot_title = 'Number of Neighborhood Services'
# plot_name = f'{access}_race_eth'

# # Organizing
# df_plot = access_2_mpo(access, metric, geography)
# display(df_plot.head())


# # Plotting
# fig = px.bar(df_plot, y='Race_Ethnicity', x=metric
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Race/Ethnicity</b>'
# fig.update_traces(hovertemplate='%{x}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=5, range=[0, 81], tickformat=',.0f')


# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left", 
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



Code graveyard

In [ ]:

# def access_1_comtype(access, metric, geography):

#     file_name = f"Community_Type_2024_dissolve__access_pop_{access}.csv"
#     file_in = path_access / file_name
#     df = pd.read_csv(file_in)

#     if access=='emp':
#         cols_access = [f'drive_{access}', f'transit_{access}', f'bike_{access}', f'walk_{access}']
#     else:
#         cols_access = [f'transit_{access}', f'bike_{access}', f'walk_{access}']

#     df = df[[geography] + cols_access]
#     df = df.groupby(geography, as_index=False).mean() # soon to be weighted average by pop in comm type?
#     df = df.melt(id_vars=geography, var_name='mode', value_name=metric)

#     df['Sort_mode'] = pd.Categorical(df['mode'], cols_access)

#     df['Sort_com'] = pd.Categorical(df[geography], [
#         'Centers and Corridors'
#         , 'Established Communities'
#         , 'Developing Communities'
#         , 'Rural Residential'
#         , 'Agricultural and Natural Lands'
#         , 'Not Identified for Growth'
#     ])

#     df = df.sort_values(['Sort_com', 'Sort_mode'], ascending=[False,True])
#     df = df.drop(['Sort_mode', 'Sort_com'], axis=1)

#     df = df.reset_index(drop = True)

#     if access=='emp':
#         conditions = [
#             df['mode'] == f'transit_{access}'
#             , df['mode'] == f'bike_{access}'
#             , df['mode'] == f'walk_{access}'
#             , df['mode'] == f'drive_{access}'
#         ]
#         choices = ['Public Transit', 'Bike', 'Walk', 'Drive']
#     else:
#         conditions = [
#             df['mode'] == f'transit_{access}'
#             , df['mode'] == f'bike_{access}'
#             , df['mode'] == f'walk_{access}'
#         ]
#         choices = ['Public Transit', 'Bike', 'Walk']

#     df['mode'] = np.select(conditions, choices, default='no')

#     return df



# def access_1_juris(access, metric, geography):

#     file_name = f"City_County__access_pop_{access}.csv"
#     file_in = path_access / file_name
#     df = pd.read_csv(file_in)

#     if access=='emp':
#         cols_access = [f'drive_{access}', f'transit_{access}', f'bike_{access}', f'walk_{access}']
#     else:
#         cols_access = [f'transit_{access}', f'bike_{access}', f'walk_{access}']

#     df = df[[geography] + cols_access]
#     df = df.melt(id_vars=geography, var_name='mode', value_name=metric)

#     df['Sort_mode'] = pd.Categorical(df['mode'], cols_access)

#     df['Sort_juris'] = pd.Categorical(df[geography], [

#         'Placerville'
#         , 'South Lake Tahoe'
#         , 'El Dorado County'
        
#         , 'Auburn'
#         , 'Colfax'
#         , 'Lincoln'
#         , 'Loomis'
#         , 'Rocklin'
#         , 'Roseville'
#         , 'Placer County'

#         , 'Citrus Heights'
#         , 'Elk Grove'
#         , 'Folsom'
#         , 'Galt'
#         , 'Isleton'
#         , 'Rancho Cordova'
#         , 'Sacramento'
#         , 'Sacramento County'

#         , 'Live Oak'
#         , 'Yuba City'
#         , 'Sutter County'

#         , 'Davis'
#         , 'West Sacramento'
#         , 'Winters'
#         , 'Woodland'
#         , 'Yolo County'

#         , 'Marysville'
#         , 'Wheatland'
#         , 'Yuba County'

#     ])

#     df = df.sort_values(['Sort_juris', 'Sort_mode'], ascending=[False, True])
#     df = df.drop(['Sort_juris', 'Sort_mode'], axis=1)

#     df = df[df[geography] != 'South Lake Tahoe']
#     df.loc[df[geography].str.contains('County'), geography] = df[df[geography].str.contains('County')][geography] + ' Unincorporated'

#     df = df.reset_index(drop=True)

#     if access=='emp':
#         conditions = [
#             df['mode'] == f'transit_{access}'
#             , df['mode'] == f'bike_{access}'
#             , df['mode'] == f'walk_{access}'
#             , df['mode'] == f'drive_{access}'
#         ]
#         choices = ['Public Transit', 'Bike', 'Walk', 'Drive']
#     else:
#         conditions = [
#             df['mode'] == f'transit_{access}'
#             , df['mode'] == f'bike_{access}'
#             , df['mode'] == f'walk_{access}'
#         ]
#         choices = ['Public Transit', 'Bike', 'Walk']

#     df['mode'] = np.select(conditions, choices, default='no')

#     return df




In [ ]:

# # Set Indicator
# indicator_name = 'TravelAccess_1'
# plot_name = 'county'
# geography = 'name'

# color_map = {
#        'Walk':'#FBB117'
#        , 'Bike': '#9DC209'
#        , 'Public Transit':"#1E90FF"
#        , 'Drive':"#1F45FC"
# }


# ## Number of Jobs ---

# access = 'emp'
# metric = 'number_of_jobs'
# plot_title = 'Number of Jobs'
# plot_name = f'{access}_county'

# # Organizing
# df_plot_emp = access_1_county(access, metric, geography)
# display(df_plot_emp.head())

# # Plotting
# fig = px.bar(df_plot_emp, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=25000, range=[0, 255000], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



# ## Number of Schools ---

# access = 'edu'
# metric = 'number_of_schools'
# plot_title = 'Number of Schools'
# plot_name = f'{access}_county'

# # Organizing
# df_plot_edu = access_1_county(access, metric, geography)
# display(df_plot_edu.head())

# # Plotting
# fig = px.bar(df_plot_edu, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=5, range=[0, 46], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



# ## Number of Neighborhood Services ---

# access = 'nonwork'
# metric = 'number_of_nonwork'
# plot_title = 'Number of Neighborhood Services'
# plot_name = f'{access}_county'

# # Organizing
# df_plot_nonwork = access_1_county(access, metric, geography)
# display(df_plot_nonwork.head())

# # Plotting
# fig = px.bar(df_plot_nonwork, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=10, range=[0, 82], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



In [ ]:

# # Set Indicator
# indicator_name = 'TravelAccess_1'
# plot_name = 'community_type'
# geography = 'Webmap_ComType'

# color_map = {
#        'Walk':'#FBB117'
#        , 'Bike': '#9DC209'
#        , 'Public Transit':"#1E90FF"
#        , 'Drive':"#1F45FC"
# }


# ## Number of Jobs ---

# access = 'emp'
# metric = 'number_of_jobs'
# plot_title = 'Number of Jobs'
# plot_name = f'{access}_community_type'

# # Organizing
# df_plot_emp = access_1_comtype(access, metric, geography)
# display(df_plot_emp.head())

# # Plotting
# fig = px.bar(df_plot_emp, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=25000, range=[0, 310000], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



# ## Number of Schools ---

# access = 'edu'
# metric = 'number_of_schools'
# plot_title = 'Number of Schools'
# plot_name = f'{access}_community_type'

# # Organizing
# df_plot_edu = access_1_comtype(access, metric, geography)
# display(df_plot_edu.head())

# # Plotting
# fig = px.bar(df_plot_edu, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=5, range=[0, 61], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



# ## Number of Neighborhood Services ---

# access = 'nonwork'
# metric = 'number_of_nonwork'
# plot_title = 'Number of Neighborhood Services'
# plot_name = f'{access}_community_type'

# # Organizing
# df_plot_nonwork = access_1_comtype(access, metric, geography)
# display(df_plot_nonwork.head())

# # Plotting
# fig = px.bar(df_plot_nonwork, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=10, range=[0, 131], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)


# # ## Together ---



# # def plot_agol(export):
# #     fig.update_layout(
# #         legend_title=None
# #         , title=title
# #         , template=template
# #         , font_family=font_family
# #         , yaxis_title=None
# #         , yaxis=dict(tickfont=dict(size=12))
# #         , xaxis=dict(tickfont=dict(size=12))
# #         )
    
# #     fig.show(config=config)
    
# #     if export:
# #         fig.write_html( file=os.path.join(path_plots, f'{indicator_name}_{plot_name}.html'), config=config)
        


# # plot_name = 'community_type'

# # df_plot_emp    ['Service'] = 'Number of Jobs'
# # df_plot_edu    ['Service'] = 'Number of Schools'
# # df_plot_nonwork['Service'] = 'Number of Neighborhood Services'

# # df_plot_emp     = df_plot_emp    .rename(columns={'number_of_jobs':'Access'})
# # df_plot_edu     = df_plot_edu    .rename(columns={'number_of_schools':'Access'})
# # df_plot_nonwork = df_plot_nonwork.rename(columns={'number_of_nonwork':'Access'})

# # df_plot = pd.concat([df_plot_emp, df_plot_edu, df_plot_nonwork])

# # display(df_plot.head())

# # # Plotting
# # fig = px.bar(df_plot, x='Access', y=geography, orientation='h'
# #               , color='mode'
# #               , color_discrete_map=color_map
# #               , facet_col = 'Service')

# # title = f'<b>Access by Modes of Transportation by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# # fig.update_traces(hovertemplate='%{y}')
# # fig.update_layout(legend={'traceorder': 'reversed'})
# # fig.update_xaxes(matches=None)
# # fig.for_each_annotation(lambda a: a.update(text=''))


# # fig.layout.xaxis.title.text = "Number of Jobs"
# # fig.layout.xaxis2.title.text = "Number of Schools"
# # fig.layout.xaxis3.title.text = "Number of Neighborhood Services"


# # # Add an annotation
# # fig.add_annotation(
# #     x=0.8, y=1.075,
# #     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
# #     showarrow=False,
# #     xanchor="left",
# #     yanchor="top",
# #     font=dict(size=10),
# #     xref="paper", yref="paper"
# # )

# # export=False
# # plot_agol(export=export)


In [ ]:

# ## Number of Schools ---

# access = 'edu'
# metric = 'number_of_schools'
# plot_title = 'Number of Schools'
# plot_name = f'{access}_jurisdiction'

# # Organizing
# df_plot = access_1_juris(access, metric, geography)
# display(df_plot.head())

# # Plotting
# fig = px.bar(df_plot, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Jurisdiction</b>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=5, range=[0, 67], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



# ## Number of Neighborhood Services ---

# access = 'nonwork'
# metric = 'number_of_nonwork'
# plot_title = 'Number of Neighborhood Services'
# plot_name = f'{access}_jurisdiction'

# # Organizing
# df_plot = access_1_juris(access, metric, geography)
# display(df_plot.head())

# # Plotting
# fig = px.bar(df_plot, x=metric, y=geography, orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map)

# title = f'<b>{plot_title} Accessible by Modes of Transportation by Jurisdiction</b>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder': 'reversed'})
# fig.update_xaxes(tick0=0, dtick=10, range=[0, 131], tickformat=',.0f')

# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.05,
#     text="*Population estimated using 2023 ACS 5-Year Estimates<br>Access estimated using 2024 Overture",
#     showarrow=False,
#     xanchor="left",
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )


# plot_agol(export=export)



In [ ]:


# # Set Indicator
# indicator_name = 'Access_1'
# plot_name = 'community_type'



# ## Importing ---

# file_name = "Community_Type_2024__access_pop_emp.csv"
# file_in = path_access / file_name
# df_com = pd.read_csv(file_in)


# ## Organizing ---
# df_plot = df_com.copy()

# df_plot = df_plot[['Webmap_ComType', 'transit_emp', 'bike_emp', 'walk_emp', 'drive_emp']]
# df_plot = df_plot.groupby('Webmap_ComType', as_index=False).mean() # soon to be weighted average by pop in comm type
# df_plot = df_plot.reset_index(drop = True)


# display(df_plot.head())


# ## Plotting ---

# fig = px.bar(df_plot, x='Webmap_ComType', y='transit_emp')
# fig.update_traces(marker_color='#1E90FF')

# title = f'<b>Transit Accessibility by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(showlegend = False)
# fig.update_yaxes(tick0=0, dtick=5000, range=[0, 51000], tickformat=',.0f')


# plot_agol(export=export)



In [ ]:


# # Set Indicator
# indicator_name = 'Access_1'
# plot_name = 'community_type'

# ## Importing ---

# file_name = "Community_Type_2024__access_pop_emp.csv"
# file_in = path_access / file_name
# df_com = pd.read_csv(file_in)


# ## Organizing ---
# df_plot = df_com.copy()

# df_plot = df_plot[['Webmap_ComType', 'transit_emp', 'bike_emp', 'walk_emp', 'drive_emp']]
# df_plot = df_plot.groupby('Webmap_ComType', as_index=False).mean() # soon to be weighted average by pop in comm type
# df_plot = df_plot.melt(id_vars='Webmap_ComType', var_name='mode', value_name='accessibility_score')

# df_plot['Sort_mode'] = pd.Categorical(df_plot['mode'], [
#     'drive_emp'
#     , 'transit_emp'
#     , 'bike_emp'
#     , 'walk_emp'
# ])

# df_plot['Sort_com'] = pd.Categorical(df_plot['Webmap_ComType'], [
#     'Centers and Corridors'
#        , 'Established Communities'
#        , 'Developing Communities'
#        , 'Rural Residential'
#        , 'Agricultural and Natural Lands'
#        , 'Not Identified for Growth'
# ])

# df_plot = df_plot.sort_values(['Sort_com', 'Sort_mode'])
# df_plot = df_plot.drop(['Sort_mode', 'Sort_com'], axis=1)

# df_plot = df_plot.reset_index(drop = True)

# display(df_plot.head())


# ## Plotting ---

# color_map = {
#        'Not Identified for Growth':'#7E587E'
#        , 'Agricultural and Natural Lands':'#DC381F'
#        , 'Rural Residential': '#FBB117'
#        , 'Developing Communities':"#9DC209"
#        , 'Established Communities':"#1E90FF"
#        , 'Centers and Corridors':'#1F45FC'

# }


# fig = px.bar(df_plot, x='mode', y='accessibility_score'
#               , color='Webmap_ComType'
#               , color_discrete_map=color_map)

# title = f'<b>Travel Accessibility by Community Type</b>  <br><sup>6-County Sacramento Region</sup>'
# fig.update_traces(hovertemplate='%{y}')
# fig.update_layout(legend={'traceorder':'reversed'})
# fig.update_yaxes(tick0=0, dtick=100000, range=[0, 725000], tickformat=',.0f')


# plot_agol(export=export)



In [ ]:


# # Set Indicator
# indicator_name = 'TravelAccess_2'
# plot_name = 'race_eth_by_county'


# ## Importing ---

# file_name1 = "tl_2020_sacog_county__access_asian_emp.csv"
# file_name2 = "tl_2020_sacog_county__access_black_emp.csv"
# file_name3 = "tl_2020_sacog_county__access_white_emp.csv"
# file_name4 = "tl_2020_sacog_county__access_hispanic_emp.csv"


# file_in1 = path_access / file_name1
# file_in2 = path_access / file_name2
# file_in3 = path_access / file_name3
# file_in4 = path_access / file_name4


# df_eth1 = pd.read_csv(file_in1)
# df_eth2 = pd.read_csv(file_in2)
# df_eth3 = pd.read_csv(file_in3)
# df_eth4 = pd.read_csv(file_in4)


# df_eth1['Race_Ethnicity'] = 'Asian (NH)'
# df_eth2['Race_Ethnicity'] = 'Black or African American (NH)'
# df_eth3['Race_Ethnicity'] = 'White (NH)'
# df_eth4['Race_Ethnicity'] = 'Hispanic or Latino'

# df_eth = pd.concat([df_eth1, df_eth2, df_eth3, df_eth4])


# ## Organizing ---

# df_plot = df_eth.copy()

# df_plot = df_plot[['name', 'Race_Ethnicity', 'transit_emp', 'bike_emp', 'walk_emp', 'drive_emp']]
# df_plot = df_plot.melt(id_vars=['name', 'Race_Ethnicity'], var_name='mode', value_name='number_of_jobs')

# df_plot['Sort_mode'] = pd.Categorical(df_plot['mode'], [
#     'drive_emp'
#     , 'transit_emp'
#     , 'bike_emp'
#     , 'walk_emp'
# ])

# df_plot['Sort_eth'] = pd.Categorical(df_plot['Race_Ethnicity'], [
#     'Asian (NH)'
#     , 'Black or African American (NH)'
#     , 'Hispanic or Latino'
#     , 'White (NH)'
# ])

# df_plot = df_plot.sort_values(['name', 'Sort_eth', 'Sort_mode'], ascending=[True, False, True])
# df_plot = df_plot.drop(['Sort_eth', 'Sort_mode'], axis=1)

# df_plot = df_plot.reset_index(drop=True)

# conditions = [
#     df_plot['mode'] == 'transit_emp'
#     , df_plot['mode'] == 'bike_emp'
#     , df_plot['mode'] == 'walk_emp'
#     , df_plot['mode'] == 'drive_emp'
# ]

# choices = ['Public Transit', 'Bike', 'Walk', 'Drive']
# df_plot['mode'] = np.select(conditions, choices, default='no')


# display(df_plot.head())


# ## Plotting ---

# color_map = {
#        'Walk':'#FBB117'
#        , 'Bike': '#9DC209'
#        , 'Public Transit':"#1E90FF"
#        , 'Drive':"#1F45FC"
# }

# fig = px.bar(df_plot, x='number_of_jobs', y='Race_Ethnicity', orientation='h'
#               , color='mode'
#               , color_discrete_map=color_map
#               , facet_row='name')

# title = f'<b>Number of Jobs Accessible by Modes of Transportation by Race/Ethnicity by County</b>'
# fig.update_traces(hovertemplate='%{x}')
# fig.update_layout(legend=dict(orientation="h", yanchor="bottom", y=-0.15,xanchor="right", x=0.4))
# fig.for_each_yaxis(lambda x: x.update(title=''))
# fig.for_each_annotation(lambda a: a.update(text=re.sub('name=', '', a.text)))
# for annotation in fig['layout']['annotations']: 
#     annotation['textangle']=0
# fig.update_xaxes(tick0=0, dtick=25000, range=[0, 265000], tickformat=',.0f')



# # Add an annotation
# fig.add_annotation(
#     x=0.9, y=-0.1,
#     text="*Population estimated using 2023 ACS<br>Jobs estimated using 2020 data",
#     showarrow=False,
#     xanchor="left", 
#     yanchor="top",
#     font=dict(size=10),
#     xref="paper", yref="paper"
# )




# plot_agol(export=export)

